# 4.2 Demo - Deploying a Simple DAB
**Lesson:** Introduction to Declarative Automation Bundles (DABs)

A bundle is a folder with a `databricks.yml` at its root:
- **`bundle`**: the project name
- **`include`**: more YAML files, usually one per resource
- **`resources`**: jobs, pipelines, dashboards, models, schemas, volumes, apps, ...
- **`targets`**: where and how to deploy (dev, stage, prod)

![Bundle anatomy: folder tree, databricks.yml top-level keys, resource file and source file](../images/4.2-bundle-anatomy.svg)

**How to read it:** left is the folder printed by the next cell, middle are the top-level keys of its `databricks.yml`, right is the job
file pulled in by `include`. The job's `notebook_path` points back to `src/hello.py`; dashed keys are optional and appear
from 4.4 on. The purple `targets` block is where dev, stage and prod differ.

Lifecycle: `init` → edit → `validate` → `deploy` → `run` → `destroy`.

![Bundle lifecycle: validate, deploy, run, destroy and what each does in the workspace](../images/4.2-bundle-lifecycle.svg)

**How to read it:** the top row is sections A-D of this notebook, run on your machine. `deploy` writes three things to the dev target:
the files, the state used to compute the next diff, and the `[dev <you>]` job. `run` starts that job; `destroy`
(dashed) removes all of it. `validate` changes nothing.

In [ ]:
from pathlib import Path
from adelab import bundles
from adelab.spark import PROJECT_ROOT

BUNDLE = PROJECT_ROOT / "04-automated-deployment-dabs" / "bundles" / "01_simple_dab"
status = bundles.workspace_status()
bundles.tree(BUNDLE)

In [ ]:
for f in ("databricks.yml", "resources/hello_job.yml", "src/hello.py"):
    print(f"──── {f}\n{(BUNDLE / f).read_text()}")

## A. Validate
`databricks bundle validate` loads all files, applies the target, resolves `${...}` references, and checks the result against the schema. Offline, `validate_schema()` uses the same JSON schema (`databricks bundle schema`), and `preview()` shows the resolved configuration.

In [ ]:
bundles.validate_schema(BUNDLE)
_ = bundles.preview(BUNDLE, "dev")

Notice what `mode: development` did: the job name got a `[dev <you>]` prefix, so every developer can deploy the same bundle side by side without collisions.

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "validate", cwd=BUNDLE)

## B. Deploy
Uploads the bundle's files (every file in the folder that Git doesn't ignore, unless `sync` says otherwise) to `/Workspace/Users/<you>/.bundle/simple_dab/dev/files` and creates or updates the job. Run it again after a change: only the differences are applied.

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "deploy", cwd=BUNDLE)
    bundles.cli("bundle", "summary", cwd=BUNDLE)

## C. Run
`bundle run <resource key>` starts the job and waits for it to finish. Open the printed URL to see it in the Jobs UI.

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "run", "hello_job", cwd=BUNDLE)

## D. Destroy
Removes everything this bundle deployed (job and files). Use it for dev targets you no longer need.

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "destroy", "--auto-approve", cwd=BUNDLE)